# Member A: Pipeline lead

**Tasks**

- [x] Verify nflverse column names against the plan (section 1)
- [x] `src/utils.py`, `pull_nfl.py`, `build_master.py`
- [ ] Match drafted players to college records; match rate by position group and draft year (sections 4-5)
- [ ] List unmatched players (section 6)

Reusable code lives in `src/data/build_master.py`; this notebook only calls it.

In [1]:
import sys; sys.path.append('..')
import pandas as pd
from src.utils import *
from src.data.build_master import match_report, unmatched_players

master = pd.read_parquet(PROCESSED / 'master.parquet')
print(master.shape)
master.head()

(3264, 55)


,season,round,pick,team,gsis_id,pfr_id,cfb_player_id,player_name,hof,pos_draft,...,shuttle,pos_combine,has_combine,pos_group,name_clean,pick_tier,outcome,has_college_match,has_college_stats,college_match_method
0,2010,1,1,STL,00-0027854,BradSa00,sam-bradford-1,Sam Bradford,False,QB,...,NaN,QB,True,QB,sam bradford,1-5,44.0,False,False,none
1,2010,1,2,DET,00-0027855,SuhxNd99,ndamukong-suh-1,Ndamukong Suh,False,DT,...,4.44,DT,True,IDL,ndamukong suh,1-5,100.0,False,False,none
2,2010,1,3,TAM,00-0027856,McCoGe99,gerald-mccoy-1,Gerald McCoy,False,DT,...,4.48,DT,True,IDL,gerald mccoy,1-5,69.0,False,False,none
3,2010,1,4,WAS,00-0027857,WillTr21,trent-williams-2,Trent Williams,False,T,...,4.63,OT,True,OT,trent williams,1-5,109.0,False,False,none
4,2010,1,5,KAN,00-0027858,BerrEr99,eric-berry-1,Eric Berry,False,DB,...,4.23,S,True,S,eric berry,1-5,50.0,False,False,none


## 1. Schema check: plan vs. actual
The plan's section 7 names versus what `build_master.py` really produces.

In [2]:
PLAN_COLS = ['season','round','pick','team','player_name','name_clean','pfr_id','gsis_id',
             'pos_raw','pos_group','college','age','outcome',
             'height','weight','forty','vertical','broad_jump','bench','cone','shuttle',
             'has_combine','has_college_match']
missing = [c for c in PLAN_COLS if c not in master.columns]
print('In plan but not in master:', missing)
print('Extra columns:', [c for c in master.columns if c not in PLAN_COLS])

In plan but not in master: ['height']
Extra columns: ['cfb_player_id', 'hof', 'pos_draft', 'category', 'side', 'to', 'allpro', 'probowls', 'seasons_started', 'w_av', 'car_av', 'dr_av', 'games', 'nfl_pass_completions', 'nfl_pass_attempts', 'nfl_pass_yards', 'nfl_pass_tds', 'nfl_pass_ints', 'nfl_rush_atts', 'nfl_rush_yards', 'nfl_rush_tds', 'nfl_receptions', 'nfl_rec_yards', 'nfl_rec_tds', 'nfl_def_solo_tackles', 'nfl_def_ints', 'nfl_def_sacks', 'pos_players', 'height_in', 'pos_combine', 'pick_tier', 'has_college_stats', 'college_match_method']


## 2. Row counts and ID completeness

In [3]:
print(master.groupby('season').size().to_string())
print('missing pfr_id:', master.pfr_id.isna().sum(), '| missing gsis_id:', master.gsis_id.isna().sum())
print('missing outcome:', master.outcome.isna().sum())

season
2010    252
2011    252
2012    247
2013    250
2014    253
2015    254
2016    248
2017    249
2018    249
2019    249
2020    249
2021    255
2022    257
missing pfr_id: 10 | missing gsis_id: 24
missing outcome: 235


## 3. Position groups

In [4]:
print(master.pos_group.value_counts(dropna=False).to_string())
master[master.pos_group.isna()][['season','pick','player_name','pos_raw','college']]

pos_group
LB      402
WR      402
CB      382
EDGE    359
IOL     335
RB      298
S       273
IDL     243
OT      225
TE      197
QB      148


,season,pick,player_name,pos_raw,college


## 4. College match rate by position group

In [5]:
by_pos, by_year, methods = match_report(master)
by_pos

,n,id_match,stats_match
pos_group,,,
CB,382,0.0,0.0
EDGE,359,0.0,0.0
IDL,243,0.0,0.0
IOL,335,0.0,0.0
LB,402,0.0,0.0
OT,225,0.0,0.0
QB,148,0.0,0.0
RB,298,0.0,0.0
S,273,0.0,0.0


## 5. College match rate by draft year

In [6]:
by_year

,n,id_match,stats_match
season,,,
2010,252,0.0,0.0
2011,252,0.0,0.0
2012,247,0.0,0.0
2013,250,0.0,0.0
2014,253,0.0,0.0
2015,254,0.0,0.0
2016,248,0.0,0.0
2017,249,0.0,0.0
2018,249,0.0,0.0


In [7]:
methods  # pick = matched on (season, overall pick); name = fallback; none = unmatched

college_match_method
none    3264
Name: count, dtype: int64

## 6. Unmatched players

In [8]:
un = unmatched_players(master)
FIGURES.parent.mkdir(exist_ok=True)
un.to_csv(FIGURES.parent / 'unmatched_college.csv', index=False)
print(len(un), 'unmatched')
un.groupby('college').size().sort_values(ascending=False).head(15)

3264 unmatched


college
Alabama        107
LSU             91
Ohio St.        88
Georgia         81
Florida         75
Oklahoma        70
Clemson         66
Notre Dame      60
USC             59
Miami (FL)      59
Michigan        58
Penn St.        57
Florida St.     57
Stanford        49
Wisconsin       48
dtype: int64

## 7. Findings / what it means
_Fill in after the real run: which positions and years match worst, whether misses are small-school (FCS) players, and the OL point: OL should match on ID but have no stats rows._